<a href="https://colab.research.google.com/github/yashnkapadia/ece324-TANGO/blob/PIRA/PIRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Inputs: The model will take in scenario descriptors, specifically demand changes that follow road closures or transit maintenance.
#Outputs: The model needs to predict SUMO-level performance metrics.
#Specifically, it must output an updated time-of-day timing plan.
#Outputs: It must also output a scenario impact summary covering delay, throughput, and queues based on SUMO evaluation outputs.

#Use supervised regression

#Training Data:
#Training follows a curriculum approach where the trained ASCE controller is
#evaluated across progressively more complex demand and infrastructure
#perturbations; PIRA is then trained on those resulting scenario-outcome pairs.

In [ ]:
import xml.etree.ElementTree as ET
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import Linear

In [ ]:
def parse_sumo_network(xml_file_path):
    """
    Parses a SUMO network XML file to create a PyTorch Geometric edge_index.
    XML file should contain a list of junctions as nodes and roads as edges.
    """
    #Load & parse the XML file
    tree = ET.parse(xml_file_path)
    root = tree.getroot()

    #Map string junction IDs to integer indices (PyG requirement apparently)
    node_id_to_idx = {}
    current_idx = 0

    #Find all junctions to register as nodes
    for junction in root.findall('junction'):
        j_id = junction.get('id')
        j_type = junction.get('type')

        #Only map valid junctions. (Filtering also possible by j_type == 'traffic_light' if you only want signalized intersections in the graph)
        if j_id and j_id not in node_id_to_idx:
            node_id_to_idx[j_id] = current_idx
            current_idx += 1

    #Extract edges (roads connecting the junctions)
    source_nodes = []
    target_nodes = []

    for edge in root.findall('edge'):
        from_node = edge.get('from')
        to_node = edge.get('to')

        #Only keep edges that connect known junctions (ignores internal/dummy edges)
        if from_node in node_id_to_idx and to_node in node_id_to_idx:
            source_nodes.append(node_id_to_idx[from_node])
            target_nodes.append(node_id_to_idx[to_node])

    #Create the PyTorch Geometric edge_index tensor
    edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)

    return edge_index, node_id_to_idx

#--- Usage memo ---
#Upload 'osm.net.xml' to Colab first
#edge_index, node_mapping = parse_sumo_network('osm.net.xml')
#edge_index: 2D tensor, [2, num_edges]
#print(f"Created graph with {len(node_mapping)} nodes and {edge_index.shape[1]} edges.")
#print(edge_index)

In [ ]:
def create_graph_dataset(parquet_path, edge_index, node_id_to_idx):
    """
    Reads the Parquet logs and builds a list of PyG Data objects.
    """
    df = pd.read_parquet(parquet_path)
    dataset = []

    #Group by scenario and time step so each graph is basically one snapshot
    for (scenario, time_step), group in df.groupby(['scenario_id', 'time_step']):
        group = group.sort_values('intersection_id')

        #Extract features (x) - Note: Update these columns to match Parquet later
        node_features = group[['queue_ns', 'queue_ew', 'avg_speed_ns', 'avg_speed_ew']].values
        x_tensor = torch.tensor(node_features, dtype=torch.float)

        #Extract targets (y) - delay, throughput, queue_total
        targets = group[['delay', 'throughput', 'queue_total']].values
        y_tensor = torch.tensor(targets, dtype=torch.float)

        graph_data = Data(x=x_tensor, edge_index=edge_index, y=y_tensor)
        dataset.append(graph_data)

    return dataset

In [ ]:
class PIRAModel(torch.nn.Module):
    def __init__(self, num_node_features, num_output_targets):
        super(PIRAModel, self).__init__()
        #Message Passing Layers
        self.conv1 = GCNConv(num_node_features, 64)
        self.conv2 = GCNConv(64, 32)
        #Regression Output Layer
        self.out = Linear(32, num_output_targets)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        predictions = self.out(x)
        return predictions

In [ ]:
def train_pira(model, dataloader, optimizer, criterion, epochs=50):
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        for batch in dataloader:
            optimizer.zero_grad()
            predictions = model(batch)
            loss = criterion(predictions, batch.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs} | Training MSE Loss: {avg_loss:.4f}")

In [ ]:
def evaluate_pira(model, dataloader):
    """
    Evaluates the model against the TANGO proposal's success conditions.
    """
    model.eval() # Set model to evaluation mode

    all_preds = []
    all_targets = []

    with torch.no_grad(): #Disable gradient tracking for evaluation
        for batch in dataloader:
            predictions = model(batch)
            all_preds.append(predictions)
            all_targets.append(batch.y)

    #Concatenate all batches
    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    #Calculate Mean Absolute Percentage Error
    #Also adding a tiny epsilon (1e-8) to avoid dividing by zero
    epsilon = 1e-8
    mape = torch.mean(torch.abs((all_targets - all_preds) / (all_targets + epsilon))) * 100

    #Calculate R-squared
    target_mean = torch.mean(all_targets, dim=0)
    ss_tot = torch.sum((all_targets - target_mean) ** 2)
    ss_res = torch.sum((all_targets - all_preds) ** 2)
    r2 = 1 - (ss_res / ss_tot)

    print("\n--- PIRA Evaluation Results ---")
    print(f"MAPE: {mape.item():.2f}% (Target: <= 10.00%)")
    print(f"R-squared: {r2.item():.4f} (Target: >= 0.85)")
    return mape.item(), r2.item()

In [ ]:

#Parse the network map
edge_index, node_map = parse_sumo_network('osm.net.xml')

#Create the dataset from parquet file (change name later)
dataset = create_graph_dataset('data/final/dataset.parquet', edge_index, node_map)

train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset, batch_size=32, shuffle=False)

#Initialize Model, Optimizer, and Loss
model = PIRAModel(num_node_features=4, num_output_targets=3)
optimizer = Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

#Train the Model
train_pira(model, train_loader, optimizer, criterion, epochs=50)

#Evaluate the Model
evaluate_pira(model, test_loader)
